In [ ]:
import pandas as pd

# Replace with the actual path to your CSV file in Drive
file_path = './CLO_SAF_DATA(Sheet1).csv'

df_original = pd.read_csv(file_path , encoding='latin1')
print(df_original.head())

In [ ]:
df_original['id'] = df_original.index + 1
print(df_original.head())

In [3]:
def create_undersampled_negatives_df(arg):
  only_score_one = arg[arg["Score"] == 1]
  undersampled_df = arg[arg['Score'] == 0].sample(n=len(only_score_one), random_state=42)
  new_df = pd.concat([only_score_one, undersampled_df], ignore_index=True)
  new_df = new_df.sample(frac=1, random_state=42).reset_index(drop=True)
  new_df = new_df.copy(deep=True)
  return new_df

In [4]:
from sklearn.model_selection import train_test_split

def create_train_test_eval_df(df):
  train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)

  # Second split: eval vs test (50/50 of the 20% temp → 10% each)
  eval_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

  #print(train_df.head())
  print(len(train_df))
  print(len(train_df[train_df["Score"] == 1]))
  print(len(test_df))
  print(len(eval_df))

  return train_df.copy(deep=True), eval_df.copy(deep=True), test_df.copy(deep=True)

In [5]:
import time
import json
import json5

In [6]:
def get_list_of_clo__saf(test_df):
    test_df = test_df.copy(deep=True)
    search_string = "Learning Objective"
    matching_col = next((col for col in test_df.columns if search_string.lower() in col.lower()), None)
    test_df = test_df.rename(columns={matching_col: 'CLO'})
    search_string = "SAF"
    matching_col = next((col for col in test_df.columns if search_string.lower() in col.lower()), None)
    test_df = test_df.rename(columns={matching_col: 'Accreditation_Standard'})
    List_of_clo_saf_pairs = test_df[["CLO", "Accreditation_Standard", "id"]].to_dict(orient="records")
    return List_of_clo_saf_pairs

In [7]:
def parse_llm_response_to_json(response):
    try:
        if type(response) == list:
            response = response[0]
        text = response['generated_text']
        # print("Raw LLM response:", text)
        clean_text = text.strip()
        # print(clean_text, type(clean_text))
        if clean_text.count('"') % 2 != 0:
            clean_text += '"'  #in case response gets cut in middle, to handle odd number of quotes
        if clean_text.startswith('{') and not clean_text.endswith('}'):
            clean_text += "}"  #in case response gets cut in middle, to handle missing closing brace
        # print(clean_text)
        parsed_json = json5.loads(clean_text)
        return parsed_json
    except Exception as e:
        raise ValueError(f"Failed to parse LLM response to JSON: {e}")

In [8]:
import json5

def parse_results_for_list_response(response):
    try:
        if type(response) == list:
            response = response[0]
        text = response['generated_text']
        test_result = json5.loads(text)
        # print(test_result)
        return test_result
    except Exception as e:
        raise ValueError(f"Failed to parse LLM response to JSON: {e}")

In [9]:
def get_llm_response_for_list(predictor, prompt, test_df, div):
  results = []
  
  List_of_clo_saf_pairs = get_list_of_clo__saf(test_df)

  for i in range(0, len(List_of_clo_saf_pairs), 10):
      listForThisRound = List_of_clo_saf_pairs[i:i + 10]
      toValue = i + 10
      my_prompt =  prompt + json.dumps(listForThisRound, ensure_ascii=False) + "\n<|im_end|>\n <|im_start|>assistant\n"
      payload = {
            "inputs": my_prompt,
            "parameters": {
                "max_new_tokens": 8000,
                "return_full_text": False,
                "stop": ["<|im_start|>", "<|im_end|>"]
            },
        }
      try:
          print("Reached response")
          # response = predictor.predict(payload, custom_attributes="accept_eula=false")
          response = predictor.predict(payload, custom_attributes="accept_eula=false",     
                                initial_args={"ContentType": "application/json", "Accept": "application/json"})

          print(response)
          parsed_response = parse_results_for_list_response(response)
          print(parsed_response)
          results.extend(parsed_response)
          print(f"{toValue}/{len(test_df)}: Done")
          
      except Exception as e:
        print(f"Error at index {i}: {e}")
      time.sleep(2) #to avoid hitting rate limit and overload error
  return results

In [10]:
def get_llm_response(predictor, prompt, test_df):
    results = []

    List_of_clo_saf_pairs = get_list_of_clo__saf(test_df)
    print(List_of_clo_saf_pairs)
    for i in range(0,len(test_df)):
        my_prompt = prompt + "\nCLO: " + List_of_clo_saf_pairs[i]['CLO'] + "\nAccreditation_Standard: " + List_of_clo_saf_pairs[i]['Accreditation_Standard'] + "\nid: " + str(List_of_clo_saf_pairs[i]['id']) + "\n<|im_end|>\n <|im_start|>assistant\n"
        print(my_prompt)
        payload = {
            "inputs": my_prompt,
            "parameters": {
                "max_new_tokens": 1000,
                "return_full_text": False,
                "stop": ["<|im_start|>", "<|im_end|>"]
            },
        }
        try:
            print("Reached response")
            response = predictor.predict(payload, custom_attributes="accept_eula=false",     
                                initial_args={"ContentType": "application/json", "Accept": "application/json"})
            print(response)
            pasrsed_response = parse_llm_response_to_json(response)
            print(pasrsed_response)
            results.append(pasrsed_response)
        except Exception as e:
            print(f"Error at index {i}: {e}")
        time.sleep(2) #to avoid hitting rate limit and overload error
    return results
                
        

In [11]:
def process_results(result_df, results, filename):
    test_df = result_df.copy(deep=True)
    test_df['pred_score'] = 0
    test_df['explanation'] = ""
    test_df['classification'] = str([])

    for i in range(len(results)):
        try:
            result_id = results[i]["id"]
            if type(result_id) == str:
                result_id = int(result_id)
            matching_row = test_df[test_df["id"] == result_id]
            if matching_row.empty:
                print(f"Error at index {i} - No matching result_id found in test_df for id: {result_id}")
                continue
            testIdx = test_df.index[test_df["id"] == result_id][0]
            response = results[i]
            if "explanation" in response:
                test_df.loc[testIdx, 'explanation'] = response["explanation"]
            if("mapLabels" in response):
                test_df.loc[testIdx, 'classification'] = str(response["mapLabels"])
                if(len(response["mapLabels"]) > 0):
                    test_df.loc[testIdx, 'pred_score'] = 1
                else:
                    test_df.loc[testIdx, 'pred_score'] = 0
            else:
                test_df.loc[testIdx, 'classification'] = str([])
                test_df.loc[testIdx, 'pred_score'] = 0
        except Exception as e:
            print(f"Error at index {i}: {e} \n {results[i]} \n testIdx: {testIdx}")
    test_df.to_csv(filename, index=False)
    return test_df

In [12]:
def get_accuracy_result(result_df):
    result_df = result_df.copy(deep=True)
    accuracy = sum(result_df["pred_score"] == result_df["Score"]) / len(result_df)

    New_matching = 0 # original score=0 but predicted label = 1
    Match_for_score_0 = 0
    Match_for_score_1 = 0
    missed_match = 0 #original score = 1 but predicted label = 0

    for i in range(len(result_df)):
        row = result_df.iloc[i]
        if row['Score'] == 1:
            if row['pred_score'] == 1:
                Match_for_score_1 += 1
            else:
                missed_match += 1
        else:
            if row['pred_score'] == 1:
                New_matching += 1
            else:
                Match_for_score_0 += 1
    print(f"Test Accuracy: {accuracy:.3f}")
    print(f"New Matching (original score=0 but predicted label = 1): {New_matching}")
    print(f"Match for score 0: {Match_for_score_0}")
    print(f"Match for score 1: {Match_for_score_1}")
    print(f"Missed Match (original score = 1 but predicted label = 0): {missed_match}")


In [13]:
import ast

def parse_classification_string(classification_str):
    try:
        labels = ast.literal_eval(classification_str)
        flat_list = []
        for item in labels:
            # Only split if item is a string containing ','
            if isinstance(item, str) and ',' in item:
                flat_list.extend([x.strip() for x in item.split(',')])
            else:
                flat_list.append(item)
        filtered_labels = [label.upper() for label in flat_list if label.upper() in {'I', 'R', 'E'}]
        return list(set(filtered_labels))
    except Exception as e:
        print(f"Error parsing classification string: {classification_str} - {e}")
        return list()
        


In [14]:
def process_label_results(df):
    full_match = 0
    partial_match = 0
    complete_mismatch = 0
    newlabelIntroduced = 0

    for i in range(len(df)):
        row = df.iloc[i]
        string_version_pred_labels = parse_classification_string(row['classification'])
        actual_labels = row[['I', 'R', 'E']].tolist()
        clean_labels = [v for v in actual_labels if pd.notna(v)]

        if len(string_version_pred_labels) == 0 and len(clean_labels) != 0:
            complete_mismatch += 1
        elif len(string_version_pred_labels) != 0 and len(clean_labels) == 0:
            complete_mismatch += 1
        elif all(elem in string_version_pred_labels for elem in clean_labels):
            full_match += 1
        elif any(elem in string_version_pred_labels for elem in clean_labels):
            partial_match += 1
        else:
            complete_mismatch += 1

        if any(elem not in clean_labels for elem in string_version_pred_labels):
            newlabelIntroduced += 1


    accuracy= round((full_match+partial_match)/(full_match+partial_match+complete_mismatch), 4)
    new_label_prop = round(newlabelIntroduced/(full_match+partial_match+complete_mismatch), 4)

    print(f"All Labels Match: {full_match}")
    print(f"At Least 1 Label Matches: {partial_match}")
    print(f"No labels Match: {complete_mismatch}")
    print(f"Accuracy with at least 1 correct label prediction: {accuracy} ({round(accuracy*100, 2)}%)")
    print(f"Proportion in which new labels are introduced: {new_label_prop} ({round(new_label_prop*100, 2)}%)")

In [15]:
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import hamming_loss

def calculate_hamming_loss(df):
    true_labels = []
    pred_labels = []
    for i in range(len(df)):
        row = df.iloc[i]
        actual_labels = row[['I', 'R', 'E']].tolist()
        clean_labels = [v for v in actual_labels if pd.notna(v)]
        # print(clean_labels)
        true_labels.append(list(set(clean_labels)))

        filtered_labels = parse_classification_string(row['classification'])
        # print("filtered labels:", filtered_labels, 'original labels:', clean_labels)
        pred_labels.append(filtered_labels)
        # print(string_version_pred_labels)
    mlb = MultiLabelBinarizer()
    mlb.fit(true_labels)
    labels_true_bin = mlb.transform(true_labels)
    labels_pred_bin = mlb.transform(pred_labels)
    print("Label order used:", mlb.classes_)
    # print("True labels:", labels_true_bin)
    # print("Pred Labels:", labels_pred_bin)
    loss = hamming_loss(labels_true_bin, labels_pred_bin)
    print("Hamming loss:", loss)

In [16]:
case1_undersampled_negatives_df = create_undersampled_negatives_df(df_original)
case1_train_df, case1_eval_df, case1_test_df = create_train_test_eval_df(case1_undersampled_negatives_df)
# print(case1_test_df.head())

454
223
57
57


In [ ]:
import sagemaker
print(dir(sagemaker))

In [18]:
import os
from dotenv import load_dotenv

load_dotenv()

ACCESS_KEY = os.getenv("ACCESS_KEY")
SECRET_KEY = os.getenv("SECRET_KEY")
SAGEMAKER_ROLE_ARN = os.getenv("IAM_ROLE_ARN")



In [19]:
import boto3

boto_session = boto3.Session(
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name= "us-west-2"
)

In [20]:
from sagemaker.jumpstart.model import JumpStartModel
from sagemaker.session import Session


In [21]:
sagemaker_session = Session(boto_session=boto_session)

In [ ]:
endpoints = sagemaker_session.sagemaker_client.list_endpoints()
print(endpoints)

## Start Model Setup

In [24]:
from sagemaker.jumpstart.notebook_utils import list_jumpstart_models
from sagemaker.jumpstart.filters import And

text_generation_models = list_jumpstart_models(sagemaker_session=sagemaker_session) 

In [25]:
own_models = [m for m in text_generation_models if "qwen" in m]

print(f"Models containing 'qwen': {len(own_models)}")
for model_id in own_models:
    print(model_id)

Models containing 'qwen': 6
huggingface-llm-qwen2-0-5b
huggingface-llm-qwen2-0-5b-instruct
huggingface-llm-qwen2-1-5b
huggingface-llm-qwen2-1-5b-instruct
huggingface-llm-qwen2-7b
huggingface-llm-qwen2-7b-instruct


In [26]:
from sagemaker import instance_types

instance_type = instance_types.retrieve(
    model_id="huggingface-llm-qwen2-7b-instruct",
    model_version="*",
    scope="inference",
    sagemaker_session=sagemaker_session)
print(instance_type)

default_instance_type = instance_types.retrieve_default(
    model_id="huggingface-llm-qwen2-7b-instruct",
    model_version="*",
    scope="inference",
    sagemaker_session=sagemaker_session)
print(default_instance_type)

Using model 'huggingface-llm-qwen2-7b-instruct' with wildcard version identifier '*'. You can pin to version '1.2.7' for more stable results. Note that models may have different input/output signatures after a major version upgrade.


['ml.g5.2xlarge', 'ml.g5.4xlarge', 'ml.g5.8xlarge', 'ml.g5.12xlarge', 'ml.g5.16xlarge', 'ml.g5.24xlarge', 'ml.g5.48xlarge', 'ml.p4d.24xlarge']
ml.g5.12xlarge


In [68]:
(
    model_id,
    model_version,
) = (
    "huggingface-llm-qwen2-7b-instruct",
    "*",
)

In [69]:
vllm_config = {
    "OPTION_MAX_MODEL_LEN": "8000",
}

In [70]:
from sagemaker.jumpstart.model import JumpStartModel


model = JumpStartModel(model_id=model_id, model_version=model_version,
                       sagemaker_session=sagemaker_session,
                       role=SAGEMAKER_ROLE_ARN,
                       env=vllm_config
                       )

In [ ]:
print(model)
print(model.env)

In [34]:
print(sagemaker.__version__)

2.218.0


In [72]:
predictor = model.deploy(
    initial_instance_count=1,
    instance_type='ml.g5.2xlarge',
    endpoint_name="huggingface-llm-qwen2-7b-instruct-modified-max-model-len",
    accept_eula=True,
    container_startup_health_check_timeout=900
    )

-----------------!

## Case 1: Individual CLO-Accreditation Standard pairs

In [73]:
case1_prompt = ''' <|im_start|>system \n
You are an experienced course designer who aligns Course Learning Outcomes (CLOs) with accreditation standards.
You follow instructions precisely and always return responses in JSON format for given CLO and accreditation standard pair, the JSON 
must be the following object (not wrapped in any other key):
                {
                    "id": <given id for CLO and accreditation standard pair>,
                     "CLO": "<given CLO>",
                     "Accreditation_Standard": "<given accreditation standard>",
                    "is_mapped": true or false,
                    "mapLabels": ["I", "R", "E"],
                    "explanation": "<explanation phrase>",
                } 
Note: A CLO may span multiple mapping scale levels. If so, classify it under all relevant levels.
<|im_end|>\n <|im_start|>user \n
An instructor has developed Course Learning Outcomes (CLOs) for a course syllabus and seeks assistance in aligning each CLO with 
each accreditation standard and categorizing its cognitive level according to the given mapping scale
As an experienced course designer:\n 1. Compare the given CLO with the given accreditation
standard to determine if there is alignment.\n 2. If they align, give an explanation for why they align. \n
3. If they align, classify the CLO into one or more levels of the given mapping scale:
Mapping Scale Levels: \n
I — Intoduce \n
R — Reinforce \n
E — Emphasize/Apply \n
Please review the following CLO and accreditation standard and complete the alignment, explanation, and mapping level determination:'''


In [ ]:
case1_results = get_llm_response(predictor, case1_prompt, case1_test_df)

In [79]:
case1_df_with_predictions = process_results(case1_test_df, case1_results, './Test(qwen2-7b)WithID.csv')

In [80]:
get_accuracy_result(case1_df_with_predictions)

Test Accuracy: 0.789
New Matching (original score=0 but predicted label = 1): 12
Match for score 0: 14
Match for score 1: 31
Missed Match (original score = 1 but predicted label = 0): 0


In [81]:
process_label_results(case1_df_with_predictions)

All Labels Match: 28
At Least 1 Label Matches: 16
No labels Match: 13
Accuracy with at least 1 correct label prediction: 0.7719 (77.19%)
Proportion in which new labels are introduced: 0.6667 (66.67%)


In [82]:
calculate_hamming_loss(case1_df_with_predictions)

Label order used: ['E' 'I' 'R']
Hamming loss: 0.40350877192982454


## Case 2: Test for list CLO and Accreditation Standards Pairs

In [83]:
case2_prompt = ''' <|im_start|>system\n
You are an experienced course designer who aligns Course Learning Outcomes (CLOs) with accreditation standards.
You follow instructions precisely and always return responses in JSON format.
Return ONLY valid JSON.
The output MUST be a JSON ARRAY, even if there is only one item.
Do not include any text outside JSON.
Do not use markdown or wrap the array in any key.
For each CLO and accreditation standard pair in the list, the JSON must be the following object (not wrapped in any other key):
                {
                    "id": <given id for CLO and accreditation standard pair>,
                     "CLO": "<given CLO>",
                     "Accreditation_Standard": "<given accreditation standard>",
                    "is_mapped": true or false,
                    "mapLabels": ["I", "R", "E"],
                    "explanation": "<explanation phrase>",
                } 
Note: A CLO may span multiple mapping scale levels. If so, classify it under all relevant levels.
<|im_end|>\n <|im_start|>user\n
An instructor has developed Course Learning Outcomes (CLOs) for a course syllabus and seeks assistance in aligning each CLO with 
each accreditation standard and categorizing its cognitive level according to the given mapping scale
As an experienced course designer:\n 1. Compare the given CLO with the given accreditation
standard to determine if there is alignment.\n 2. If they align, give an explanation for why they align. \n
3. If they align, classify the CLO into one or more levels of the given mapping scale:
Mapping Scale Levels: \n
I — Intoduce \n
R — Reinforce \n
E — Emphasize/Apply \n
Please review the following list of CLO and accreditation standard and complete the alignment, explanation, and mapping level determination:'''



In [84]:
case2_undersampled_negatives_df = create_undersampled_negatives_df(df_original)
case2_train_df, case2_eval_df, case2_test_df = create_train_test_eval_df(case2_undersampled_negatives_df)
# print(case1_test_df.head())

454
223
57
57


In [ ]:
case2_results = get_llm_response_for_list(predictor, case2_prompt, case2_test_df, int(len(case2_test_df)/12))

In [90]:
case2_df_with_predictions = process_results(case2_test_df, case2_results, './Test(qwen2-7b)ListOfSentencesWithID.csv')

In [91]:
get_accuracy_result(case2_df_with_predictions)

Test Accuracy: 0.772
New Matching (original score=0 but predicted label = 1): 10
Match for score 0: 16
Match for score 1: 28
Missed Match (original score = 1 but predicted label = 0): 3


In [92]:
process_label_results(case2_df_with_predictions)

All Labels Match: 30
At Least 1 Label Matches: 11
No labels Match: 16
Accuracy with at least 1 correct label prediction: 0.7193 (71.93%)
Proportion in which new labels are introduced: 0.5965 (59.65%)


In [93]:
calculate_hamming_loss(case2_df_with_predictions)

Label order used: ['E' 'I' 'R']
Hamming loss: 0.38011695906432746


# Clean up 

In [ ]:
# Delete the SageMaker endpoint
all_models = sagemaker_session.sagemaker_client.list_models()
print(all_models)

response = sagemaker_session.sagemaker_client.list_endpoint_configs()
print(response)

endpoints = sagemaker_session.sagemaker_client.list_endpoints()
print(endpoints)

predictor.delete_model()
predictor.delete_endpoint()

In [ ]:
## Delete Artifacts and their associations

all_artifacts = sagemaker_session.sagemaker_client.list_artifacts()
print(all_artifacts)

for artifact in all_artifacts['ArtifactSummaries']:
    all_associations = sagemaker_session.sagemaker_client.list_associations(SourceArn=artifact['ArtifactArn'])
    print(all_associations)
    for association in all_associations['AssociationSummaries']:
        sagemaker_session.sagemaker_client.delete_association(SourceArn=association['SourceArn'], 
                                                              DestinationArn=association['DestinationArn'])
    sagemaker_session.sagemaker_client.delete_artifact(ArtifactArn=artifact['ArtifactArn'])
    

In [ ]:
endpoints = sagemaker_session.sagemaker_client.list_endpoints()
print(endpoints)

In [101]:
def delete_all_sagemaker_resources():
    all_objects = list(locals().values()) + list(globals().values())
    deletable_objects = [obj for obj in all_objects if hasattr(obj, 'delete') and obj.__class__.__module__ == 'sagemaker.core.resources']
    
    print(deletable_objects)
    for obj in deletable_objects:
        obj.delete()
        
delete_all_sagemaker_resources()

[]


In [ ]:
all_models = sagemaker_session.sagemaker_client.list_models()
print(all_models)

for model_info in all_models['Models']:
    model_name = model_info['ModelName']
    sagemaker_session.sagemaker_client.delete_model(ModelName=model_name)
    
all_models = sagemaker_session.sagemaker_client.list_models()
print(all_models)

In [ ]:
sagemaker_session.sagemaker_client.close()

In [ ]:
del sagemaker_session
del boto_session